In [45]:
import torch
from IPython import display
from d2l import d2l_torch as d2l
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

训练数据集已存在，直接读取...
测试数据集已存在，直接读取...


### 3.6.1 初始化模型参数

In [46]:
num_inputs = 28 * 28
num_outputs = 10
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

### 3.6.2 定义softmax操作

In [47]:
def softmax(X):
    print(f"X.shape = {X.shape}")
    print(f"X[:10] = {X[:10]}")
    X_exp = torch.exp(X)
    print(f"X_exp.shape = {X_exp.shape}")
    print(f"X_exp[:10] = {X_exp[:10]}")
    partition = X_exp.sum(1, keepdim=True)
    print(f"partition.shape = {partition.shape}")
    print(f"partition[:10] = {partition[:10]}")
    return X_exp / partition

In [48]:
X = torch.normal(0, 1, (2, 5))
X_prob = softmax(X)
X_prob, X_prob.sum(1)

X.shape = torch.Size([2, 5])
X[:10] = tensor([[-0.5929, -0.5829, -0.7888,  1.1858, -1.0529],
        [-1.0013, -1.2765,  2.0971, -0.3155, -1.0482]])
X_exp.shape = torch.Size([2, 5])
X_exp[:10] = tensor([[0.5527, 0.5583, 0.4544, 3.2733, 0.3489],
        [0.3674, 0.2790, 8.1428, 0.7294, 0.3506]])
partition.shape = torch.Size([2, 1])
partition[:10] = tensor([[5.1877],
        [9.8691]])


(tensor([[0.1065, 0.1076, 0.0876, 0.6310, 0.0673],
         [0.0372, 0.0283, 0.8251, 0.0739, 0.0355]]),
 tensor([1.0000, 1.0000]))

### 3.6.3 定义模型

In [49]:
def net(X):
    # W.shape = (784, 10)
    # X.reshape((-1, W.shape[0])), X为输入特征，形状通常是(batch_size, 28, 28), reshape之后的形状为(batch_size, 784)
    # torch.matmul(..., W): 矩阵乘法，reshape之后的X与权重矩阵相乘，形状为(batch_size, 784)的矩阵 乘以 形状为(784, num_classes), 相乘后的结果形状为(batch_size, num_classes)
    # 最后加上偏置向量b，形状为(num_classes,), 利用广播机制，会自动将偏置加到每个样本上，相加后的形状为(batch_size, num_classes)
    # 最后返回softmax的值
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

### 3.6.4 定义损失函数

In [50]:
y = torch.tensor([0, 2])
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])
y_hat[[0, 1], y]        # 高级索引的写法，输出y_hat中第0行和第1行，然后取出每一行对应样本的真实标签列（第0列和第2列）
# y_hat[torch.arange(len(y_hat)), y]        # 可以用这种写法替换，这个写法效率最高
# y_hat[range(len(y_hat)), y]               # 这种写法性能次之

tensor([0.1000, 0.5000])

In [51]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])

cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

### 3.6.5 分类精度

In [52]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        # 如果y_hat是二维的概率分布，
        # 例如y_hat = [[0.1, 0.3, 0.6],
        #               [0.3, 0.2, 0.5]]
        # 取每一行最大值的索引，y_hat.argmax(1) = [2, 2]
        y_hat = y_hat.argmax(1)
    # 先将y_hat的值转换类型，与y的数据类型一致
    # y_hat = y_hat.type(y.dtype)
    cmp = y_hat.type(y.dtype) == y
    # 运算之后的cmp的数据类型为布尔类型 cmp = [True, False, False, ..., True]
    print(cmp)
    # 先将cmp的布尔类型转换成int类型: 0或1，然后再求和，求和之后再转换成float类型，方便后续计算精确度
    return float(cmp.type(y.dtype).sum())

In [53]:
accuracy(y_hat, y)

tensor([False,  True])


1.0

In [54]:
accuracy(y_hat, y) / len(y)

tensor([False,  True])


0.5

In [55]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [56]:
def evaluate_accuracy(net, data_iter):
    """计算在指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval()      # 将模型设置为评估模式
    metric = Accumulator(2)     # 正确预测数、预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [57]:
evaluate_accuracy(net, test_iter)

X.shape = torch.Size([256, 10])
X[:10] = tensor([[-0.2128,  0.0205,  0.0864, -0.0296, -0.0829, -0.0513, -0.0516,  0.1070,
         -0.0439,  0.0024],
        [-0.2121,  0.0914,  0.2045,  0.1168,  0.1697, -0.0599, -0.0453, -0.0936,
         -0.1191, -0.3259],
        [-0.3094, -0.2501,  0.2930,  0.3317, -0.0361, -0.1172, -0.2089, -0.0469,
         -0.2146, -0.2366],
        [-0.1641, -0.1662,  0.1361,  0.1545, -0.0055, -0.0728, -0.0842, -0.0051,
         -0.1703, -0.2301],
        [-0.2296, -0.0573,  0.1954,  0.0172,  0.0390, -0.1118, -0.0078, -0.0399,
         -0.0466, -0.1012],
        [-0.4203, -0.1360,  0.2402,  0.2948,  0.0079, -0.0922, -0.2003, -0.0513,
         -0.2084, -0.0894],
        [-0.1580, -0.0359, -0.0118,  0.1394, -0.0063,  0.0328, -0.0464, -0.1349,
         -0.0965, -0.0377],
        [-0.0961, -0.0653,  0.1312,  0.1655,  0.0126,  0.0087,  0.0111,  0.0108,
         -0.1107, -0.1136],
        [-0.0346,  0.0151,  0.0400,  0.0068,  0.0608, -0.0202, -0.0316,  0.0421,
      

0.0843

### 3.6.6 训练

In [58]:
def train_epoch_ch3(net, train_iter, loss, updater):
    """训练模型一轮"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()

    # 训练损失总和、训练准确度总和、样本数
    metric = Accumulator(3)
    for X, y in train_iter:
        # 计算梯度并更新参数
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            # 使用PyTorch内置的优化器和损失函数
            updater.zero_grad()
            l.mean().backward()
            updater.step()
        else:
            # 使用定制的优化器和损失函数
            l.sum().backward()
            updater(X.shape[0])
        metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    # 返回训练损失和训练精度
    return metric[0] / metric[2], metric[1] / metric[2]